In [1]:
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())
openai_api_key = os.environ["OPENAI_API_KEY"]

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

In [3]:
from typing import Optional

from langchain_core.pydantic_v1 import BaseModel, Field

class Kisi(BaseModel):
    """kisi ile ilgili bilgiler."""

    # ^ Doc-string for the entity Person.
    # This doc-string is sent to the LLM as the description of the schema Person,
    # and it can help to improve extraction results.

    # Note that:
    # 1. Each field is an `optional` -- this allows the model to decline to extract it!
    # 2. Each field has a `description` -- this description is used by the LLM.
    # Having a good description can help improve extraction results.
    isim: Optional[str] = Field(
        default=None, description="Eğer biliniyorsa kişi nin adı"
    )
    soyisim: Optional[str] = Field(
        default=None, description="Eğer biliniyorsa kişi nin soyadı"
    )
    ulke: Optional[str] = Field(
        default=None, description="Eğer biliniyorsa kişi nin ülkesi"
    )

## Extractor Tanımlama

In [4]:
from typing import Optional

from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.pydantic_v1 import BaseModel, Field

# Define a custom prompt to provide instructions and any additional context.
# 1) You can add examples into the prompt template to improve extraction quality
# 2) You can introduce additional parameters to take context into account (e.g., include metadata
#    about the document from which the text was extracted.)
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an expert extraction algorithm. "
            "Only extract relevant information from the text. "
            "If you do not know the value of an attribute asked to extract, "
            "return null for the attribute's value.",
        ),
        ("human", "{text}"),
    ]
)

In [5]:
chain = prompt | llm.with_structured_output(schema=Kisi)

In [6]:
comment = "Selam ben Halit Türker. Merzifonluyum. bilgisayar mühendisliği 4 sınıf da okuyorum . Ülkemi çok seviyorum"

In [7]:
chain.invoke({"text": comment})

Kisi(isim='Halit', soyisim='Türker', ulke='Türkiye')

## Extraction list

In [8]:
from typing import List, Optional

from langchain_core.pydantic_v1 import BaseModel, Field


class Kisi(BaseModel):
    """kisi ile ilgili bilgiler."""

    # ^ Doc-string for the entity Person.
    # This doc-string is sent to the LLM as the description of the schema Person,
    # and it can help to improve extraction results.

    # Note that:
    # 1. Each field is an `optional` -- this allows the model to decline to extract it!
    # 2. Each field has a `description` -- this description is used by the LLM.
    # Having a good description can help improve extraction results.
    isim: Optional[str] = Field(
        default=None, description="Eğer biliniyorsa kişi nin adı"
    )
    soyisim: Optional[str] = Field(
        default=None, description="Eğer biliniyorsa kişi nin soyadı"
    )
    ulke: Optional[str] = Field(
        default=None, description="Eğer biliniyorsa kişi nin ülkesi"
    )
    
class Data(BaseModel):
    """Kisi ile ilgili Extracted data."""

    # Creates a model so that we can extract multiple entities.
    Kisiler: List[Kisi]

In [9]:
chain = prompt | llm.with_structured_output(schema=Data)

In [10]:
comment = "Selam ben Halit Türker. Merzifonluyum. bilgisayar mühendisliği 4 sınıf da okuyorum . Ülkemi çok seviyorum"

In [11]:
chain.invoke({"text": comment})

Data(Kisiler=[Kisi(isim='Halit', soyisim='Türker', ulke='Türkiye')])

In [12]:
# Example input text that mentions multiple people
comment = "Selam ben Halit Türker. Merzifonluyum. bilgisayar mühendisliği 4 sınıf da okuyorum . Ülkemi çok seviyorum. Yanımda eşim Necla var.Eşim olduğu için benimle aynı soyadını taşıyor o paris doğumlu ve orada yaşıyor"
#Not aynı soyadını demeseydik soyadı na none diyecekti

# Invoke the processing chain on the text
response = chain.invoke({"text": comment})

# Output the extracted data
response

Data(Kisiler=[Kisi(isim='Halit', soyisim='Türker', ulke='Türkiye'), Kisi(isim='Necla', soyisim='Türker', ulke='Fransa')])